In [1]:
# ============================================================
# Script purpose:
# This notebook updates Metasheet_Analysis.xlsx by filling
# Patient_Identifier and Patient_id.
#
# 1. Patient_Identifier is extracted from Collaborator_sample_id
#    Example: AUP-AL1Y-NB1 -> AL1Y
#
# 2. Clinical Barcode values from AUP_ClinicalData_20260609.NO.DA
#    are used to build a mapping between Patient_Identifier
#    and numeric patient_id.
#    Example: AUP-AQ0Z-TTP2-B -> AQ0Z -> patient_id = 1
#
# 3. Patient_id is filled in the metasheet when the extracted
#    Patient_Identifier matches the clinical Barcode-derived ID.
#
# 4. The script prints mapping summaries and reports conflicts
#    where one Patient_Identifier maps to multiple patient_id values.
# ============================================================

import re
from pathlib import Path
import pandas as pd

# ============================================================
# Input files
# ============================================================

metasheet_file = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
)

clinical_file = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUP_ClinicalData_20260609.xlsx"
)

clinical_sheet = "AUP_ClinicalData_20260609.NO.DA"

# Final updated output
out_file = metasheet_file.with_name(
    "Metasheet_Analysis.with_Patient_Identifier_and_Patient_id.xlsx"
)

# If True, overwrite existing Patient_Identifier and Patient_id columns
OVERWRITE_EXISTING = True


# ============================================================
# Helper functions
# ============================================================

def is_blank(x):
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in {"nan", "na", "none", "null"}


def clean_patient_id(x):
    """
    Clean numeric patient_id from clinical file.
    Example:
        1      -> "1"
        1.0    -> "1"
        "1.0"  -> "1"
    """
    if is_blank(x):
        return pd.NA

    s = str(x).strip()

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def extract_aup_identifier(x):
    """
    Extract 4-character patient identifier.

    Examples:
        AUP-AQ0Z-TTP2-B  -> AQ0Z
        AUP-AQ0Z-TTM1-A  -> AQ0Z
        AUP-AMCZ-TTP1-B  -> AMCZ
        AMCZ-NB1         -> AMCZ
        AQ30-NB1         -> AQ30
        AO01-NB1         -> AO01
        AM01-NB1         -> AM01
    """
    if is_blank(x):
        return pd.NA

    s = str(x).strip().upper()

    # Pattern 1: AUP-XXXX-...
    m = re.search(r"\bAUP-([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    # Pattern 2: XXXX-NB1 / XXXX-NB2 etc.
    m = re.search(r"\b([A-Z0-9]{4})-NB\d+\b", s)
    if m:
        return m.group(1)

    # Fallback: first 4-character block before a dash
    m = re.search(r"\b([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    return pd.NA


# ============================================================
# 1. Read files
# ============================================================

print("Reading metasheet...")
meta = pd.read_excel(metasheet_file, sheet_name=0)

print("Reading clinical file...")
clin = pd.read_excel(clinical_file, sheet_name=clinical_sheet)

print(f"Metasheet rows: {meta.shape[0]}")
print(f"Clinical rows: {clin.shape[0]}")


# ============================================================
# 2. Check required columns
# ============================================================

required_meta_cols = [
    "Collaborator_sample_id",
    "Patient_Identifier",
    "Patient_id",
]

required_clin_cols = [
    "patient_id",
    "Barcode",
]

for col in required_meta_cols:
    if col not in meta.columns:
        raise ValueError(f"Missing column in metasheet: {col}")

for col in required_clin_cols:
    if col not in clin.columns:
        raise ValueError(f"Missing column in clinical file: {col}")


# ============================================================
# 3. Fill Patient_Identifier in metasheet
#    From Collaborator_sample_id
#    Example: AUP-AL1Y-NB1 -> AL1Y
# ============================================================

meta["Patient_Identifier_new"] = meta["Collaborator_sample_id"].apply(
    extract_aup_identifier
)

if OVERWRITE_EXISTING:
    meta["Patient_Identifier"] = meta["Patient_Identifier_new"]
else:
    mask = meta["Patient_Identifier"].apply(is_blank)
    meta.loc[mask, "Patient_Identifier"] = meta.loc[mask, "Patient_Identifier_new"]

meta = meta.drop(columns=["Patient_Identifier_new"])

print("\n========== Patient_Identifier summary ==========")
print(f"Rows with Patient_Identifier: {meta['Patient_Identifier'].notna().sum()}")
print(
    f"Unique Patient_Identifier in metasheet: "
    f"{meta['Patient_Identifier'].dropna().nunique()}"
)


# ============================================================
# 4. Build crosswalk from clinical Barcode
#    Barcode -> Patient_Identifier -> numeric patient_id
# ============================================================

clin_xwalk = clin[["patient_id", "Barcode"]].copy()

clin_xwalk["Patient_Identifier"] = clin_xwalk["Barcode"].apply(
    extract_aup_identifier
)

clin_xwalk["patient_id_clean"] = clin_xwalk["patient_id"].apply(
    clean_patient_id
)

# Keep only usable rows
clin_xwalk = clin_xwalk[
    clin_xwalk["Patient_Identifier"].notna()
    & clin_xwalk["patient_id_clean"].notna()
].copy()

print("\n========== Clinical crosswalk input summary ==========")
print(f"Clinical rows with usable Barcode + patient_id: {clin_xwalk.shape[0]}")
print(
    f"Unique Patient_Identifier from clinical Barcode: "
    f"{clin_xwalk['Patient_Identifier'].nunique()}"
)


# ============================================================
# 5. Collapse clinical crosswalk
#    If one Patient_Identifier maps to multiple patient_id values,
#    report as conflict and do not use that identifier for filling.
# ============================================================

id_to_patient_ids = (
    clin_xwalk
    .groupby("Patient_Identifier")["patient_id_clean"]
    .apply(lambda x: sorted(set(x), key=lambda y: int(y) if str(y).isdigit() else str(y)))
)

conflicts = id_to_patient_ids[id_to_patient_ids.apply(len) > 1]
valid_map_series = id_to_patient_ids[id_to_patient_ids.apply(len) == 1].apply(lambda x: x[0])
valid_map = valid_map_series.to_dict()

print("\n========== Crosswalk summary ==========")
print(f"Valid one-to-one mappings: {len(valid_map)}")
print(f"Conflicting mappings: {len(conflicts)}")

if len(conflicts) > 0:
    print("\nWARNING: These Patient_Identifier values map to multiple patient_id values.")
    print("They will NOT be used for filling Patient_id until manually checked.\n")
    conflict_df = conflicts.reset_index()
    conflict_df.columns = ["Patient_Identifier", "patient_id_values"]
    print(conflict_df.to_string(index=False))


# ============================================================
# 6. Fill Patient_id in metasheet
# ============================================================

meta["Patient_id_mapped"] = meta["Patient_Identifier"].map(valid_map)

if OVERWRITE_EXISTING:
    meta["Patient_id"] = meta["Patient_id_mapped"]
else:
    mask = meta["Patient_id"].apply(is_blank)
    meta.loc[mask, "Patient_id"] = meta.loc[mask, "Patient_id_mapped"]

meta = meta.drop(columns=["Patient_id_mapped"])

# Convert Patient_id to integer-like format when possible
patient_id_numeric = pd.to_numeric(meta["Patient_id"], errors="coerce")

if patient_id_numeric.notna().sum() == meta["Patient_id"].notna().sum():
    meta["Patient_id"] = patient_id_numeric.astype("Int64")


# ============================================================
# 7. Print final matching summary
# ============================================================

matched_rows = meta["Patient_id"].notna().sum()
matched_unique_ids = meta.loc[
    meta["Patient_id"].notna(), "Patient_Identifier"
].dropna().nunique()

unmatched_identifiers = sorted(
    meta.loc[
        meta["Patient_Identifier"].notna() & meta["Patient_id"].isna(),
        "Patient_Identifier",
    ]
    .dropna()
    .unique()
)

print("\n========== Final metasheet summary ==========")
print(f"Total metasheet rows: {meta.shape[0]}")
print(f"Rows with Patient_id filled: {matched_rows}")
print(f"Unique Patient_Identifier matched to Patient_id: {matched_unique_ids}")
print(f"Unique Patient_Identifier without Patient_id match: {len(unmatched_identifiers)}")

if len(unmatched_identifiers) > 0:
    print("\nFirst 50 unmatched Patient_Identifier values:")
    print(unmatched_identifiers[:50])


# ============================================================
# 8. Save updated metasheet
# ============================================================

meta.to_excel(out_file, index=False)

print("\nDONE")
print(f"Updated metasheet saved here:\n{out_file}")

Reading metasheet...
Reading clinical file...
Metasheet rows: 784
Clinical rows: 6728

========== Patient_Identifier summary ==========
Rows with Patient_Identifier: 784
Unique Patient_Identifier in metasheet: 270

========== Clinical crosswalk input summary ==========
Clinical rows with usable Barcode + patient_id: 1049
Unique Patient_Identifier from clinical Barcode: 267

========== Crosswalk summary ==========
Valid one-to-one mappings: 267
Conflicting mappings: 0

========== Final metasheet summary ==========
Total metasheet rows: 784
Rows with Patient_id filled: 769
Unique Patient_Identifier matched to Patient_id: 262
Unique Patient_Identifier without Patient_id match: 8

First 50 unmatched Patient_Identifier values:
['AMW4', 'ANU0', 'AOEG', 'AOHU', 'APDI', 'APLC', 'APRD', 'ARSR']

DONE
Updated metasheet saved here:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.with_Patient_Identifier_and_Patient_id.xlsx


In [3]:
import pandas as pd
from pathlib import Path

# ============================================================
# Script purpose:
# Fill Histological_subtype, ER_Status, PR_Status, HER2_Status,
# and Molecular_subtype in the updated AURORA metasheet.
#
# Matching logic:
#   Metasheet Patient_id  <-->  Clinical patient_id
#
# Clinical information is taken only from rows where:
#   Category == "First_Primary"
#
# Molecular subtype is derived from ER/PR/HER2 status.
# ============================================================


# ============================================================
# Input files
# ============================================================

metasheet_file = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.with_Patient_Identifier_and_Patient_id.xlsx"
)

clinical_file = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUP_ClinicalData_20260609.xlsx"
)

clinical_sheet = "AUP_ClinicalData_20260609.NO.DA"

# Output file
out_file = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.with_Histology_Receptor_status.xlsx"
)


# ============================================================
# Helper functions
# ============================================================

def is_blank(x):
    """Check missing or blank-like values."""
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in {"nan", "na", "none", "null", "missing"}


def clean_text(x):
    """Clean text values."""
    if is_blank(x):
        return pd.NA
    return str(x).strip()


def clean_id(x):
    """
    Clean patient IDs while preserving IDs like 588-1.

    Examples:
        1      -> "1"
        1.0    -> "1"
        588-1  -> "588-1"
    """
    if is_blank(x):
        return pd.NA

    s = str(x).strip()

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def collapse_unique(values):
    """
    Collapse values for one patient_id.

    If all non-missing values are the same:
        return that value

    If multiple different non-missing values exist:
        return pd.NA and flag as conflict later
    """
    vals = []

    for v in values:
        if not is_blank(v):
            vals.append(str(v).strip())

    unique_vals = sorted(set(vals))

    if len(unique_vals) == 0:
        return pd.NA

    if len(unique_vals) == 1:
        return unique_vals[0]

    return "CONFLICT: " + " | ".join(unique_vals)


def normalize_receptor_status(x):
    """
    Normalize receptor status to Positive / Negative / Unknown.
    """
    if is_blank(x):
        return pd.NA

    s = str(x).strip().lower()

    if s in {"positive", "pos", "+", "1", "yes"}:
        return "Positive"

    if s in {"negative", "neg", "-", "0", "no"}:
        return "Negative"

    if "positive" in s:
        return "Positive"

    if "negative" in s:
        return "Negative"

    return pd.NA


def derive_molecular_subtype(er, pr, her2):
    """
    Derive molecular subtype from ER, PR, and HER2.

    HR+ means ER Positive OR PR Positive.
    HR- means ER Negative AND PR Negative.
    """
    er = normalize_receptor_status(er)
    pr = normalize_receptor_status(pr)
    her2 = normalize_receptor_status(her2)

    if pd.isna(er) or pd.isna(pr) or pd.isna(her2):
        return pd.NA

    hr_positive = er == "Positive" or pr == "Positive"
    hr_negative = er == "Negative" and pr == "Negative"

    if hr_positive and her2 == "Negative":
        return "HR+/HER2-"

    if hr_positive and her2 == "Positive":
        return "HR+/HER2+"

    if hr_negative and her2 == "Negative":
        return "HR-/HER2-"

    if hr_negative and her2 == "Positive":
        return "HR-/HER2+"

    return pd.NA


# ============================================================
# 1. Read files
# ============================================================

print("Reading updated metasheet...")
meta = pd.read_excel(metasheet_file, sheet_name=0)

print("Reading clinical file...")
clin = pd.read_excel(clinical_file, sheet_name=clinical_sheet)

print(f"\nMetasheet rows: {meta.shape[0]}")
print(f"Clinical rows: {clin.shape[0]}")


# ============================================================
# 2. Check required columns
# ============================================================

required_meta_cols = [
    "Patient_id",
    "Histological_subtype",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Molecular_subtype",
]

required_clin_cols = [
    "patient_id",
    "Category",
    "Histologic.type",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
]

for col in required_meta_cols:
    if col not in meta.columns:
        raise ValueError(f"Missing column in metasheet: {col}")

for col in required_clin_cols:
    if col not in clin.columns:
        raise ValueError(f"Missing column in clinical file: {col}")


# ============================================================
# 3. Clean Patient_id columns
# ============================================================

meta["Patient_id_clean"] = meta["Patient_id"].apply(clean_id)
clin["patient_id_clean"] = clin["patient_id"].apply(clean_id)

print("\n========== Patient ID summary ==========")
print(f"Unique Patient_Identifier in metasheet: {meta['Patient_Identifier'].dropna().nunique()}")
print(f"Unique Patient_id in metasheet: {meta['Patient_id_clean'].dropna().nunique()}")
print(f"Unique patient_id in clinical file: {clin['patient_id_clean'].dropna().nunique()}")

# ============================================================
# 4. Keep only First_Primary clinical rows
# ============================================================

clin["Category_clean"] = clin["Category"].apply(clean_text)

clin_fp = clin[
    clin["Category_clean"].astype(str).str.strip().str.lower() == "first_primary"
].copy()

print("\n========== First_Primary clinical summary ==========")
print(f"Clinical First_Primary rows: {clin_fp.shape[0]}")
print(
    f"Unique First_Primary patient_id values: "
    f"{clin_fp['patient_id_clean'].dropna().nunique()}"
)


# ============================================================
# 5. Collapse clinical First_Primary data per patient_id
# ============================================================

clin_fp_small = clin_fp[
    [
        "patient_id_clean",
        "Histologic.type",
        "ER_Status",
        "PR_Status",
        "HER2_Status",
    ]
].copy()

clin_fp_small = clin_fp_small[clin_fp_small["patient_id_clean"].notna()].copy()

collapsed = (
    clin_fp_small
    .groupby("patient_id_clean", as_index=False)
    .agg({
        "Histologic.type": collapse_unique,
        "ER_Status": collapse_unique,
        "PR_Status": collapse_unique,
        "HER2_Status": collapse_unique,
    })
)

collapsed = collapsed.rename(
    columns={
        "patient_id_clean": "Patient_id_clean",
        "Histologic.type": "Histological_subtype_new",
        "ER_Status": "ER_Status_new",
        "PR_Status": "PR_Status_new",
        "HER2_Status": "HER2_Status_new",
    }
)


# ============================================================
# 6. Report conflicts
# ============================================================

conflict_cols = [
    "Histological_subtype_new",
    "ER_Status_new",
    "PR_Status_new",
    "HER2_Status_new",
]

conflict_mask = collapsed[conflict_cols].apply(
    lambda row: any(str(x).startswith("CONFLICT:") for x in row if not pd.isna(x)),
    axis=1
)

conflict_df = collapsed[conflict_mask].copy()

print("\n========== Conflict summary ==========")
print(f"Patient_id values with any conflict: {conflict_df.shape[0]}")

if conflict_df.shape[0] > 0:
    print("\nConflicting First_Primary values:")
    print(conflict_df.to_string(index=False))


# ============================================================
# 7. Replace conflict values with missing before filling
# ============================================================

for col in conflict_cols:
    collapsed[col] = collapsed[col].apply(
        lambda x: pd.NA if str(x).startswith("CONFLICT:") else x
    )


# ============================================================
# 8. Merge clinical data into metasheet
# ============================================================

meta_merged = meta.merge(
    collapsed,
    on="Patient_id_clean",
    how="left"
)

matched_patient_ids = meta_merged.loc[
    meta_merged["Histological_subtype_new"].notna()
    | meta_merged["ER_Status_new"].notna()
    | meta_merged["PR_Status_new"].notna()
    | meta_merged["HER2_Status_new"].notna(),
    "Patient_id_clean"
].dropna().nunique()

print("\n========== Matching summary ==========")
print(f"Unique Patient_id matched with First_Primary clinical data: {matched_patient_ids}")


# ============================================================
# 9. Fill Histology and Receptor columns
# ============================================================

meta_merged["Histological_subtype"] = meta_merged["Histological_subtype_new"]
meta_merged["ER_Status"] = meta_merged["ER_Status_new"].apply(normalize_receptor_status)
meta_merged["PR_Status"] = meta_merged["PR_Status_new"].apply(normalize_receptor_status)
meta_merged["HER2_Status"] = meta_merged["HER2_Status_new"].apply(normalize_receptor_status)


# ============================================================
# 10. Derive Molecular_subtype
# ============================================================

meta_merged["Molecular_subtype"] = meta_merged.apply(
    lambda row: derive_molecular_subtype(
        row["ER_Status"],
        row["PR_Status"],
        row["HER2_Status"],
    ),
    axis=1
)


# ============================================================
# 11. Print final summary
# ============================================================

print("\n========== Final fill summary ==========")
print(f"Total metasheet rows: {meta_merged.shape[0]}")

print(
    f"Rows with Histological_subtype filled: "
    f"{meta_merged['Histological_subtype'].notna().sum()}"
)

print(
    f"Rows with ER_Status filled: "
    f"{meta_merged['ER_Status'].notna().sum()}"
)

print(
    f"Rows with PR_Status filled: "
    f"{meta_merged['PR_Status'].notna().sum()}"
)

print(
    f"Rows with HER2_Status filled: "
    f"{meta_merged['HER2_Status'].notna().sum()}"
)

print(
    f"Rows with Molecular_subtype filled: "
    f"{meta_merged['Molecular_subtype'].notna().sum()}"
)

print("\nHistological_subtype distribution:")
print(meta_merged["Histological_subtype"].value_counts(dropna=False))

print("\nER_Status distribution:")
print(meta_merged["ER_Status"].value_counts(dropna=False))

print("\nPR_Status distribution:")
print(meta_merged["PR_Status"].value_counts(dropna=False))

print("\nHER2_Status distribution:")
print(meta_merged["HER2_Status"].value_counts(dropna=False))

print("\nMolecular_subtype distribution:")
print(meta_merged["Molecular_subtype"].value_counts(dropna=False))


# ============================================================
# 12. Print unmatched Patient_id values
# ============================================================

unmatched_patient_ids = sorted(
    meta_merged.loc[
        meta_merged["Patient_id_clean"].notna()
        & meta_merged["Histological_subtype"].isna()
        & meta_merged["ER_Status"].isna()
        & meta_merged["PR_Status"].isna()
        & meta_merged["HER2_Status"].isna(),
        "Patient_id_clean",
    ]
    .dropna()
    .unique()
)

print("\n========== Unmatched Patient_id summary ==========")
print(f"Unique Patient_id without First_Primary clinical match: {len(unmatched_patient_ids)}")

if len(unmatched_patient_ids) > 0:
    print("\nFirst 50 unmatched Patient_id values:")
    print(unmatched_patient_ids[:50])


# ============================================================
# 13. Clean temporary columns and save
# ============================================================

drop_cols = [
    "Patient_id_clean",
    "Histological_subtype_new",
    "ER_Status_new",
    "PR_Status_new",
    "HER2_Status_new",
]

meta_final = meta_merged.drop(columns=drop_cols)

meta_final.to_excel(out_file, index=False)

print("\nDONE")
print(f"Updated file saved here:\n{out_file}")

Reading updated metasheet...
Reading clinical file...

Metasheet rows: 784
Clinical rows: 6728

========== Patient ID summary ==========
Unique Patient_Identifier in metasheet: 270
Unique Patient_id in metasheet: 262
Unique patient_id in clinical file: 274

========== First_Primary clinical summary ==========
Clinical First_Primary rows: 274
Unique First_Primary patient_id values: 274

========== Conflict summary ==========
Patient_id values with any conflict: 0

========== Matching summary ==========
Unique Patient_id matched with First_Primary clinical data: 262

========== Final fill summary ==========
Total metasheet rows: 784
Rows with Histological_subtype filled: 753
Rows with ER_Status filled: 764
Rows with PR_Status filled: 762
Rows with HER2_Status filled: 742
Rows with Molecular_subtype filled: 741

Histological_subtype distribution:
Ductal                      606
Lobular                      86
Mixed Ductal and Lobular     61
NaN                          31
Name: Histologic